# Stage 2: Temporal Modeling & Prediction Head (Transformer)
In this notebook, we implement the Temporal Modeling stage of the Spatio-Temporal SAR Architecture:
1. **Time-Series Sequences**: We adapt our dataset to return sequences of length $T$ (simulating temporal data).
2. **Temporal Modeling (Transformer)**: A Transformer Encoder architecture that takes latent sequences from the pre-trained Autoencoder.
3. **Prediction Head**: A multi-output head that translates the temporal features into Water Body Segmentation Masks.
4. **Training**: End-to-end training of the temporal model.

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import rasterio
from scipy.ndimage import uniform_filter

# Set seeds
np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

### 1. Simulating Time-Series Data
Since our Sen1Floods11 subset consists of independent samples, we will synthetically group them into sequences of length $T=3$ to demonstrate the Transformer architecture.
In a real-world scenario, you would have $t_1, t_2, t_3$ images for the exact same geographical bounding box.

In [ ]:
DATA_DIR = "data/sen1floods11"
S1_DIR = os.path.join(DATA_DIR, "S1Hand")
LABEL_DIR = os.path.join(DATA_DIR, "LabelHand")

def get_files(dir_path):
    if not os.path.exists(dir_path):
        return []
    return sorted([f for f in os.listdir(dir_path) if f.endswith('.tif')])

s1_files = get_files(S1_DIR)
label_files = get_files(LABEL_DIR)

# We expect them to match up if downloaded properly from Stage 1
file_pairs = list(zip([os.path.join(S1_DIR, f) for f in s1_files], 
                      [os.path.join(LABEL_DIR, f) for f in label_files]))

# Preprocessing from Stage 1
def apply_speckle_filter(image, size=3):
    filtered_image = np.zeros_like(image)
    for band in range(image.shape[0]):
        filtered_image[band] = uniform_filter(image[band], size=size)
    return filtered_image

def preprocess_s1(data):
    vv = np.nan_to_num(data[0], nan=-25.0)
    vh = np.nan_to_num(data[1], nan=-30.0)
    stacked = np.stack([vv, vh], axis=0)
    filtered = apply_speckle_filter(stacked, size=3)
    vv_clipped = np.clip(filtered[0], -30.0, 0.0)
    vh_clipped = np.clip(filtered[1], -35.0, -5.0)
    vv_norm = (vv_clipped - (-30.0)) / 30.0
    vh_norm = (vh_clipped - (-35.0)) / 30.0
    return np.stack([vv_norm, vh_norm], axis=0)

class TimeSeriesSARDataset(Dataset):
    def __init__(self, file_pairs, seq_length=3, target_size=(256, 256)):
        self.file_pairs = file_pairs
        self.seq_length = seq_length
        self.target_size = target_size
        
        # Create sequences (overlapping sliding window)
        self.sequences = []
        for i in range(len(file_pairs) - seq_length + 1):
            self.sequences.append(file_pairs[i:i+seq_length])
            
    def __len__(self):
        return len(self.sequences)
        
    def __getitem__(self, idx):
        seq_pairs = self.sequences[idx]
        seq_s1 = []
        
        # Load all S1 images in the sequence
        for s1_path, _ in seq_pairs:
            with rasterio.open(s1_path) as src:
                s1_data = src.read()
            s1_preprocessed = preprocess_s1(s1_data)
            s1_tensor = torch.tensor(s1_preprocessed, dtype=torch.float32).unsqueeze(0)
            s1_resized = torch.nn.functional.interpolate(
                s1_tensor, size=self.target_size, mode='bilinear', align_corners=False
            ).squeeze(0)
            seq_s1.append(s1_resized)
            
        # The label is the mask for the LAST image in the sequence (Prediction Head)
        _, label_path = seq_pairs[-1]
        with rasterio.open(label_path) as src:
            label_data = src.read(1)
        label_tensor = torch.tensor(label_data, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
        label_resized = torch.nn.functional.interpolate(
            label_tensor, size=self.target_size, mode='nearest'
        ).squeeze(0).squeeze(0).long()
        
        # Shape: [seq_length, channels, H, W]
        seq_s1_tensor = torch.stack(seq_s1)
        return seq_s1_tensor, label_resized

if len(file_pairs) >= 3:
    train_size = int(0.8 * len(file_pairs))
    train_pairs = file_pairs[:train_size]
    test_pairs = file_pairs[train_size:]
    
    train_dataset = TimeSeriesSARDataset(train_pairs, seq_length=3)
    test_dataset = TimeSeriesSARDataset(test_pairs, seq_length=3)
    
    train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False)
    print(f"Created {len(train_dataset)} train sequences and {len(test_dataset)} test sequences.")
else:
    print("Not enough files downloaded yet to create sequences. Please run Stage 1 first.")


### 2. Transformer & Prediction Head
Here we define:
1. **Autoencoder (Encoder only)**: We will load the pre-trained weights from Stage 1. It acts as our spatial feature extractor.
2. **Transformer Encoder**: Captures temporal dependencies across the sequence of spatial latent vectors.
3. **Prediction Head**: Decodes the temporal representation into a pixel-level classification mask.

In [ ]:
class SpatialEncoder(nn.Module):
    """The Encoder part of the Autoencoder from Stage 1."""
    def __init__(self):
        super(SpatialEncoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(2, 16, kernel_size=3, stride=2, padding=1),
            nn.ReLU(True),
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1),
            nn.ReLU(True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.ReLU(True)
        )
    def forward(self, x):
        return self.encoder(x) # Output shape: [B, 128, 16, 16]

class TemporalTransformerPredictor(nn.Module):
    def __init__(self, seq_length=3, embed_dim=128*16*16, num_heads=8, hidden_dim=512, num_layers=2):
        super(TemporalTransformerPredictor, self).__init__()
        
        # 1. Spatial Feature Extractor (Encoder)
        self.spatial_encoder = SpatialEncoder()
        
        # Positional Encoding for time steps
        self.pos_embedding = nn.Parameter(torch.randn(1, seq_length, embed_dim))
        
        # 2. Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads, dim_feedforward=hidden_dim, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # 3. Prediction Head (Decoder for Segmentation)
        # We take the transformer output for the LAST time step and decode it back to 256x256
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1, output_padding=1), # 32x32x64
            nn.ReLU(True),
            nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1),  # 64x64x32
            nn.ReLU(True),
            nn.ConvTranspose2d(32, 16, kernel_size=3, stride=2, padding=1, output_padding=1),  # 128x128x16
            nn.ReLU(True),
            nn.ConvTranspose2d(16, 2, kernel_size=3, stride=2, padding=1, output_padding=1),   # 256x256x2 (2 classes: non-water, water)
        )

    def forward(self, x):
        # x shape: [B, SeqLen, C, H, W]
        B, S, C, H, W = x.shape
        
        # Reshape to process all spatial images through encoder
        x_reshaped = x.view(B * S, C, H, W)
        
        # Extract Spatial Features
        spatial_features = self.spatial_encoder(x_reshaped) # Shape: [B*S, 128, 16, 16]
        
        # Flatten spatial dimensions for Transformer
        sf_flat = spatial_features.view(B, S, -1) # Shape: [B, S, 128*16*16]
        
        # Add positional embedding
        sf_flat = sf_flat + self.pos_embedding
        
        # Temporal Modeling
        temporal_features = self.transformer(sf_flat) # Shape: [B, S, 128*16*16]
        
        # Take the features of the last time step for prediction
        last_step_features = temporal_features[:, -1, :] # Shape: [B, 128*16*16]
        
        # Reshape back to spatial map for decoder
        last_step_spatial = last_step_features.view(B, 128, 16, 16)
        
        # Multi-output Prediction Head (Segmentation)
        logits = self.decoder(last_step_spatial) # Shape: [B, 2, 256, 256]
        return logits

model = TemporalTransformerPredictor(seq_length=3).to(device)

# Try loading pretrained encoder weights if available
encoder_path = 'models/encoder.pth'
if os.path.exists(encoder_path):
    print("Loading pretrained spatial encoder weights...")
    model.spatial_encoder.encoder.load_state_dict(torch.load(encoder_path, weights_only=True))
    # Optional: Freeze the encoder to only train the transformer and decoder
    # for param in model.spatial_encoder.parameters():
    #     param.requires_grad = False
else:
    print("Pretrained encoder not found. Training from scratch.")


### 3. Training the Temporal Model
We train the model using CrossEntropyLoss since it's a segmentation/classification task (Water vs Non-Water). Note: Label -1 represents invalid/no-data pixels which we should ignore in the loss.

In [ ]:
# Loss and Optimizer
criterion = nn.CrossEntropyLoss(ignore_index=-1) # Ignore -1 (invalid) pixels
optimizer = optim.Adam(model.parameters(), lr=1e-4)

num_epochs = 5
print("Starting Temporal Model training...")
try:
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for seq_data, target in train_loader:
            seq_data = seq_data.to(device)
            target = target.to(device)
            
            optimizer.zero_grad()
            outputs = model(seq_data)
            
            loss = criterion(outputs, target)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * seq_data.size(0)
            
        epoch_loss = running_loss / len(train_loader.dataset)
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.5f}")
    print("Training Complete!")
    
    # Save Model
    os.makedirs('models', exist_ok=True)
    torch.save(model.state_dict(), 'models/temporal_transformer.pth')
    print("Model saved to models/temporal_transformer.pth")
except Exception as e:
    print(f"Training could not proceed: {e}")
